In [ ]:

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database("bootcamp")
session.use_schema("atp524")  # Replace with your schema.

In [ ]:
import ast
import html
import json
import re
from pathlib import Path
from IPython.display import display, HTML

# 1. Ensure we scan the top-level Workspace directory (even if notebook is in a subfolder)
cwd = Path.cwd()
workspace_dir = cwd
for parent in [cwd, *cwd.parents]:
    # Workspace root = folder named workspace(s), a git root, or the folder holding sql/ + python/
    if (parent.name.lower() in {"workspace", "workspaces"} or (parent / ".git").exists()
            or ((parent / "sql").is_dir() and (parent / "python").is_dir())):
        workspace_dir = parent
        break

# The notebook kernel mounts the workspace under a hashed folder name, so use a readable label instead
workspace_label = "this Workspace" if re.fullmatch(r"[0-9a-f]{20,}", workspace_dir.name) else workspace_dir.name

# Known descriptions for common Snowflake AI, Vector, Text, & Stage functions
KNOWN_DESCRIPTIONS = {
    # Cortex AI & LLM
    "AI_COMPLETE": ("Cortex AI", "Generates text completions or answers prompts using an LLM."),
    "COMPLETE": ("Cortex AI", "Cortex schema function for generating LLM text completions."),
    "AI_EXTRACT": ("Cortex AI", "Extracts structured fields/JSON from unstructured text or documents."),
    "EXTRACT_ANSWER": ("Cortex AI", "Finds and extracts a specific answer span from source text."),
    "AI_SUMMARIZE": ("Cortex AI", "Condenses long text passages into a concise summary."),
    "SUMMARIZE": ("Cortex AI", "Cortex schema function for summarizing text."),
    "AI_SUMMARIZE_AGG": ("Cortex AI", "Aggregates and summarizes text across multiple rows."),
    "AI_AGG": ("Cortex AI", "Aggregates insights across multiple rows using an LLM prompt."),
    "AI_TRANSLATE": ("Cortex AI", "Translates text from one language to another."),
    "TRANSLATE": ("Cortex AI", "Cortex function that translates text between languages."),
    "AI_SENTIMENT": ("Cortex AI", "Scores overall and entity-level sentiment in text."),
    "SENTIMENT": ("Cortex AI", "Returns a sentiment score (-1 to 1) for input text."),
    "AI_CLASSIFY": ("Cortex AI", "Categorizes text or images into user-defined labels."),
    "CLASSIFY_TEXT": ("Cortex AI", "Classifies text into custom categories."),
    "AI_FILTER": ("Cortex AI", "Evaluates a natural-language boolean condition on rows/text."),
    "AI_SIMILARITY": ("Cortex AI", "Computes semantic similarity directly between two inputs."),
    "AI_COUNT_TOKENS": ("Cortex AI", "Counts the number of LLM tokens in an input text/prompt."),
    "COUNT_TOKENS": ("Cortex AI", "Counts the number of tokens for a given Cortex model/function."),
    "TRY_COMPLETE": ("Cortex AI", "Like COMPLETE(), but returns NULL instead of failing on errors."),

    # Document, Audio & Text Chunking
    "AI_PARSE_DOCUMENT": ("Document AI", "Extracts text, tables, and layout from PDFs/docs in a stage."),
    "PARSE_DOCUMENT": ("Document AI", "Cortex function for parsing text and layout from staged documents."),
    "AI_TRANSCRIBE": ("Cortex AI", "Transcribes audio files from a stage into text."),
    "SPLIT_TEXT_RECURSIVE_CHARACTER": ("Text / RAG", "Splits long text into overlapping chunks for RAG embeddings."),
    "SPLIT_TEXT_MARKDOWN_HEADER": ("Text / RAG", "Splits Markdown documents into chunks by header hierarchy."),

    # Embeddings & Vector Search
    "AI_EMBED": ("Vector / RAG", "Converts text or images into vector embeddings."),
    "EMBED_TEXT_768": ("Vector / RAG", "Generates a 768-dimension vector embedding from text."),
    "EMBED_TEXT_1024": ("Vector / RAG", "Generates a 1024-dimension vector embedding from text."),
    "VECTOR_COSINE_SIMILARITY": ("Vector / RAG", "Measures cosine similarity (-1 to 1) between two vectors."),
    "VECTOR_L2_DISTANCE": ("Vector / RAG", "Calculates Euclidean (L2) distance between two vectors."),
    "VECTOR_INNER_PRODUCT": ("Vector / RAG", "Computes the inner (dot) product of two vectors."),
    "SEARCH_PREVIEW": ("Cortex Search", "Queries a Snowflake Cortex Search service for retrieval."),

    # Stage, File & Semi-Structured Functions
    "GET_PRESIGNED_URL": ("Stage / Files", "Generates a temporary HTTPS link to access a staged file."),
    "BUILD_STAGE_FILE_URL": ("Stage / Files", "Builds a permanent Snowflake-hosted URL for a staged file."),
    "BUILD_SCOPED_FILE_URL": ("Stage / Files", "Generates a scoped URL for accessing staged files safely."),
    "GET_STAGE_LOCATION": ("Stage / Files", "Returns the external cloud storage URL of a stage."),
    "DIRECTORY": ("Stage / Files", "Queries file metadata (size, last modified, URL) in a stage."),
    "PARSE_JSON": ("Semi-Structured", "Parses a JSON string into a Snowflake VARIANT object."),
    "OBJECT_CONSTRUCT": ("Semi-Structured", "Constructs a JSON/OBJECT value from key-value pairs."),
    "ARRAY_AGG": ("Semi-Structured", "Aggregates values from multiple rows into a single ARRAY."),
    "FLATTEN": ("Semi-Structured", "Explodes semi-structured arrays/objects into relational rows.")
}

# Standard SQL / Python keywords to ignore so we only surface real functions
IGNORE_TOKENS = {
    "SELECT", "FROM", "WHERE", "GROUP", "ORDER", "BY", "HAVING", "LIMIT", "JOIN",
    "LEFT", "RIGHT", "INNER", "OUTER", "CROSS", "ON", "AND", "OR", "NOT", "IN",
    "IS", "NULL", "AS", "CASE", "WHEN", "THEN", "ELSE", "END", "VALUES", "INTO",
    "SET", "TABLE", "VIEW", "STAGE", "DATABASE", "SCHEMA", "WAREHOUSE", "ROLE",
    "IF", "EXISTS", "CREATE", "REPLACE", "ALTER", "DROP", "INSERT", "UPDATE",
    "DELETE", "MERGE", "CALL", "RETURNS", "LANGUAGE", "PYTHON", "SQL", "VARCHAR",
    "STRING", "NUMBER", "INT", "INTEGER", "FLOAT", "BOOLEAN", "VARIANT", "ARRAY",
    "OBJECT", "VECTOR", "TIMESTAMP", "DATE", "COUNT", "SUM", "AVG", "MIN", "MAX",
    "COALESCE", "NULLIF", "IFF", "CAST", "TRY_CAST", "CONCAT", "LOWER", "UPPER",
    "TRIM", "SUBSTR", "SUBSTRING", "REPLACE", "SPLIT", "TO_VARCHAR", "TO_JSON",
    "CURRENT_TIMESTAMP", "CURRENT_DATE", "PRINT", "LEN", "STR", "LIST", "DICT"
}

# 2. Gather custom Python function names so we exclude them from Snowflake functions
custom_py_funcs = set()
sql_sources = []  # (relative_path, sql_text)

for path in workspace_dir.rglob("*"):
    if ".ipynb_checkpoints" in str(path) or not path.is_file():
        continue
    rel_path = str(path.relative_to(workspace_dir))

    if path.suffix == ".py":
        code = path.read_text(encoding="utf-8", errors="ignore")
        try:
            tree = ast.parse(code)
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    custom_py_funcs.add(node.name.upper())
                # Also extract string literals inside Python (where SQL queries live)
                elif isinstance(node, ast.Constant) and isinstance(node.value, str):
                    if any(kw in node.value.upper() for kw in ("SELECT ", "CALL ", "CORTEX", "AI_", "VECTOR_", "GET_PRESIGNED")):
                        sql_sources.append((rel_path, node.value))
        except SyntaxError:
            pass

    elif path.suffix == ".sql":
        sql_sources.append((rel_path, path.read_text(encoding="utf-8", errors="ignore")))

    elif path.suffix == ".ipynb":
        try:
            nb = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            for cell in nb.get("cells", []):
                if cell.get("cell_type") != "code":
                    continue
                src = "".join(cell.get("source", []))
                lang = cell.get("metadata", {}).get("language", "")
                if lang == "sql" or src.lstrip().startswith("%%sql"):
                    sql_sources.append((rel_path, src))
                else:
                    try:
                        tree = ast.parse(src)
                        for node in ast.walk(tree):
                            if isinstance(node, ast.FunctionDef):
                                custom_py_funcs.add(node.name.upper())
                            elif isinstance(node, ast.Constant) and isinstance(node.value, str):
                                if any(kw in node.value.upper() for kw in ("SELECT ", "CALL ", "CORTEX", "AI_", "VECTOR_", "GET_PRESIGNED")):
                                    sql_sources.append((rel_path, node.value))
                    except SyntaxError:
                        pass
        except Exception:
            pass

# 3. Dynamically extract all function calls in SQL files and embedded SQL strings
# Captures optional schema prefixes like SNOWFLAKE.CORTEX.FUNC_NAME(
func_pattern = re.compile(r"\b(?:SNOWFLAKE\.CORTEX\.)?([A-Z][A-Z0-9_]{2,})\s*\(", re.IGNORECASE)

discovered = {}
for rel_path, text in sql_sources:
    # Strip SQL single-line comments (-- ...) so commented-out examples don't cause false positives
    cleaned_text = re.sub(r"--.*", "", text)
    for match in func_pattern.finditer(cleaned_text):
        fn_upper = match.group(1).upper()
        if fn_upper in IGNORE_TOKENS or fn_upper in custom_py_funcs:
            continue
        discovered.setdefault(fn_upper, set()).add(rel_path)

# 4. Build HTML table + Markdown file
rows_html = []
md_lines = [
    f"### All Snowflake & SQL Functions Discovered in {workspace_label} ({len(discovered)} Found)",
    "| Function | Category | Purpose / Definition | Used In |",
    "| :--- | :--- | :--- | :--- |"
]

for fn_name in sorted(discovered.keys(), key=lambda k: (k not in KNOWN_DESCRIPTIONS, k)):
    files = sorted(discovered[fn_name])
    if fn_name in KNOWN_DESCRIPTIONS:
        cat, desc = KNOWN_DESCRIPTIONS[fn_name]
        desc_html = html.escape(desc)
    else:
        cat = "Other / Unmapped"
        desc_html = "<span style='color: #b45309;'><em>[Needs Definition] Discovered in SQL — paste list for explanation</em></span>"
        desc = "[Needs Definition] Discovered in SQL"

    fn_html = f"<code>{html.escape(fn_name)}()</code>"
    cat_html = f"<strong>{html.escape(cat)}</strong>"
    used_html = "<br>".join(f"<code>{html.escape(f)}</code>" for f in files)

    rows_html.append(f"""
        <tr>
            <td style="width: 25%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{fn_html}</td>
            <td style="width: 15%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{cat_html}</td>
            <td style="width: 35%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{desc_html}</td>
            <td style="width: 25%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{used_html}</td>
        </tr>""")

    used_md = "<br>".join(f"`{f}`" for f in files)
    md_lines.append(f"| `{fn_name}()` | **{cat}** | {desc.replace('|', '\\|')} | {used_md} |")

table_html = f"""
<h3>All Snowflake &amp; SQL Functions Discovered in {html.escape(workspace_label)} ({len(discovered)} Found)</h3>
<table style="width: 100%; table-layout: fixed; border-collapse: collapse; font-size: 13px;">
    <thead>
        <tr style="background: #f3f4f6; text-align: left;">
            <th style="width: 25%; padding: 8px;">Function</th>
            <th style="width: 15%; padding: 8px;">Category</th>
            <th style="width: 35%; padding: 8px;">Purpose / Definition</th>
            <th style="width: 25%; padding: 8px;">Used In</th>
        </tr>
    </thead>
    <tbody>{''.join(rows_html)}
    </tbody>
</table>
"""
display(HTML(table_html))

# 5. Save the Markdown version alongside the notebook
md_path = cwd / "snowflake_functions_discovered.md"
md_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print(f"Scanned {len(sql_sources)} SQL sources; found {len(discovered)} functions. Markdown saved to {md_path.name}")
